In [1]:
import utils
import numpy as np
import plotly.graph_objects as go
import pandas as pd
from plotly.subplots import make_subplots
from sklearn_extra.cluster import KMedoids
from joblib import dump
import os

import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

# Energy Data

In [2]:
energy_df = pd.read_csv('energy_df.csv')
# energy_df = utils.get_energy_df()
# energy_df = utils.clean_data(energy_df)
# energy_df.to_csv('energy_df.csv', index=False)
print(energy_df)


               LCLid                 tstp  energy(kWh/hh)
0          MAC000002  2012-10-12 00:30:00           0.000
1          MAC000002  2012-10-12 01:00:00           0.000
2          MAC000002  2012-10-12 01:30:00           0.000
3          MAC000002  2012-10-12 02:00:00           0.000
4          MAC000002  2012-10-12 02:30:00           0.000
...              ...                  ...             ...
147202065  MAC005019  2014-02-27 22:00:00           0.129
147202066  MAC005019  2014-02-27 22:30:00           0.095
147202067  MAC005019  2014-02-27 23:00:00           0.061
147202068  MAC005019  2014-02-27 23:30:00           0.054
147202069  MAC005019  2014-02-28 00:00:00           0.074

[147202070 rows x 3 columns]


In [4]:
example_household = 'MAC003144'

household_data = (
    energy_df[energy_df['LCLid'] == example_household]
    .sort_values('tstp')     # make sure time is sorted
    .iloc[:240]              # take only first 240 rows
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=household_data['tstp'],
        y=household_data['energy(kWh/hh)'],
        mode='lines',
        name=example_household
    )
)

fig.update_layout(
    title=f"Energy Consumption for {example_household} (First 240 timestamps)",
    xaxis_title="Timestamp",
    yaxis_title="Energy (kWh/hh)"
)

fig.update_layout(
    font=dict(size=18),          # global font (axes, title, etc.)
    legend=dict(font=dict(size=18)),
)

fig.update_layout(
    width=1000,   # make it less wide (try 600–900)
    height=450   # optional
)

fig.show()


In [6]:
id_num = energy_df['LCLid'].nunique()
print(id_num)

4871


In [7]:
# Remove LCLid = 001654 from the dataframe since it is not a valid household
energy_df = energy_df[energy_df['LCLid'] != 'MAC000165']
energy_df = energy_df[energy_df['LCLid'] != 'MAC001150']
energy_df = energy_df[energy_df['LCLid'] != 'MAC005556']
energy_df = energy_df[energy_df['LCLid'] != 'MAC005559']
energy_df = energy_df[energy_df['LCLid'] != 'MAC005560']
energy_df = energy_df[energy_df['LCLid'] != 'MAC005563']
print(energy_df.shape)
print(energy_df.head())

(147162394, 3)
       LCLid                 tstp  energy(kWh/hh)
0  MAC000002  2012-10-12 00:30:00             0.0
1  MAC000002  2012-10-12 01:00:00             0.0
2  MAC000002  2012-10-12 01:30:00             0.0
3  MAC000002  2012-10-12 02:00:00             0.0
4  MAC000002  2012-10-12 02:30:00             0.0


Following LCLid only have one row of data, so discard them
- MAC001150
- MAC005556
- MAC005559
- MAC005560
- MAC005563

In [8]:
energy_df['gradient'] = abs(energy_df.groupby('LCLid')['energy(kWh/hh)'].diff().shift(-1))
mean_gradient_df = energy_df.groupby('LCLid')['gradient'].mean().reset_index()

energy_df.drop(columns=['gradient'], inplace=True)
print(mean_gradient_df.head())

       LCLid  gradient
0  MAC000002  0.109262
1  MAC000003  0.176724
2  MAC000005  0.057801
3  MAC000007  0.105829
4  MAC000008  0.100872


In [9]:
households = energy_df['LCLid'].unique()
# A sliding window of size 6 and step 2 to smooth the data for all households

window_size = 12
step_size = 1

def rolling_window_with_step(group):
    return group.rolling(window=window_size, min_periods=1, center = True).mean()[::step_size]

# Apply the rolling window with step to each group
smoothed_energy_df = energy_df.groupby('LCLid')['energy(kWh/hh)'].apply(rolling_window_with_step).reset_index()
smoothed_energy_df = smoothed_energy_df.merge(energy_df.reset_index(), left_on=['LCLid', 'level_1'], right_on=['LCLid', 'index'], suffixes=('_smoothed', ''))
smoothed_energy_df = smoothed_energy_df[['LCLid', 'tstp', 'energy(kWh/hh)_smoothed']]

# Display the first few rows of the smoothed dataframe to verify the transformation
smoothed_energy_df.head()

,LCLid,tstp,energy(kWh/hh)_smoothed
0,MAC000002,2012-10-12 00:30:00,0.0
1,MAC000002,2012-10-12 01:00:00,0.0
2,MAC000002,2012-10-12 01:30:00,0.0
3,MAC000002,2012-10-12 02:00:00,0.0
4,MAC000002,2012-10-12 02:30:00,0.0


In [10]:
example_household = 'MAC003144'

household_data = (
    smoothed_energy_df[smoothed_energy_df['LCLid'] == example_household]
    .sort_values('tstp')     # make sure time is sorted
    .iloc[:240]              # take only first 240 rows
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=household_data['tstp'],
        y=household_data['energy(kWh/hh)_smoothed'],
        mode='lines',
        name=example_household
    )
)

fig.update_layout(
    title=f"Smoothed Energy Consumption for {example_household} (First 240 timestamps)",
    xaxis_title="Timestamp",
    yaxis_title="Energy (kWh/hh)"
)

fig.update_layout(
    font=dict(size=18),          # global font (axes, title, etc.)
    legend=dict(font=dict(size=18)),
)

fig.update_layout(
    width=1000,   # make it less wide (try 600–900)
    height=450   # optional
)

fig.show()

In [22]:
# households = energy_df['LCLid'].unique()
# # Randomly select 5 households
# example_households = np.random.choice(households, 5)

# fig = go.Figure()

# for household in example_households:
#     df_household = energy_df[energy_df['LCLid'] == household]
#     fig.add_trace(go.Scatter(x=df_household['tstp'], y=df_household['energy(kWh/hh)'], mode='lines', name=household))

# fig.update_layout(title='Energy consumption for 5 example households', xaxis_title='Date and Time', yaxis_title='Energy (kWh/hh)')
# fig.show()

In [23]:
# # example_households = np.random.choice(households, 5)
# fig = go.Figure()

# for household in example_households:
#     df_household = smoothed_energy_df[smoothed_energy_df['LCLid'] == household]
#     fig.add_trace(go.Scatter(x=df_household['tstp'], y=df_household['energy(kWh/hh)_smoothed'], mode='lines', name=household))

# fig.update_layout(title='Smoothed energy consumption for 5 example households', xaxis_title='Date and Time', yaxis_title='Energy (kWh/hh)')
# fig.update_yaxes(range=[0, 1.5])
# fig.show()

# Feature Selection

In [24]:
# See if the smoothed data has any NaN values
print(smoothed_energy_df.isna().sum())

LCLid                      0
tstp                       0
energy(kWh/hh)_smoothed    0
dtype: int64


In [25]:
# Get the mean, median, std, min, max of the energy consumption for each household
energy_stats = smoothed_energy_df.groupby('LCLid')['energy(kWh/hh)_smoothed'].agg(['mean', 'median', 'std', 'min', 'max'])

# Turn it into a dataframe 
energy_stats = energy_stats.reset_index()
energy_stats = energy_stats.merge(mean_gradient_df, on='LCLid', how='left')

energy_stats.head()

,LCLid,mean,median,std,min,max,gradient
0,MAC000002,0.252475,0.212667,0.158780,0.000000,1.654250,0.109262
1,MAC000003,0.397265,0.239333,0.452749,0.023083,3.211750,0.176724
2,MAC000005,0.095424,0.079083,0.062499,0.010417,0.878667,0.057801
3,MAC000007,0.197888,0.170333,0.141215,0.038500,1.905833,0.105829
4,MAC000008,0.363182,0.334083,0.157363,0.069083,1.324083,0.100872


In [26]:
print(energy_stats.isna().sum())

LCLid       0
mean        0
median      0
std         0
min         0
max         0
gradient    0
dtype: int64


In [27]:
# Find LCLid groups with NaN standard deviation
nan_std_groups = energy_stats[energy_stats['std'].isna()].index.tolist()

# Inspect the original DataFrame for these groups
for idx in nan_std_groups:
    print(energy_stats['LCLid'][idx])

In [28]:
# Cluster the households based on the energy consumption statistics
# Select the features
X = energy_stats[['mean', 'median', 'std', 'min', 'max', 'gradient']]

In [29]:
# Twenty different colors for the clusters
red = 'rgb(255, 0, 0)'
blue = 'rgb(0, 0, 255)'
green = 'rgb(0, 255, 0)'
yellow = 'rgb(255, 255, 0)'
orange = 'rgb(255, 165, 0)'
purple = 'rgb(128, 0, 128)'
cyan = 'rgb(0, 255, 255)'
magenta = 'rgb(255, 0, 255)'
lime = 'rgb(0, 255, 0)'
teal = 'rgb(0, 128, 128)'
pink = 'rgb(255, 192, 203)'
brown = 'rgb(165, 42, 42)'
navy = 'rgb(0, 0, 128)'
olive = 'rgb(128, 128, 0)'
maroon = 'rgb(128, 0, 0)'
gray = 'rgb(128, 128, 128)'
silver = 'rgb(192, 192, 192)'
gold = 'rgb(255, 215, 0)'
ivory = 'rgb(255, 255, 240)'
indigo = 'rgb(75, 0, 130)'




colors = [red, blue, green, yellow, orange, purple, cyan, magenta, lime, teal, pink, brown, navy, olive, maroon, gray, silver, gold, ivory, indigo]

# K-Medoid Clustering

In [30]:
print(X)

          mean    median       std       min       max  gradient
0     0.252475  0.212667  0.158780  0.000000  1.654250  0.109262
1     0.397265  0.239333  0.452749  0.023083  3.211750  0.176724
2     0.095424  0.079083  0.062499  0.010417  0.878667  0.057801
3     0.197888  0.170333  0.141215  0.038500  1.905833  0.105829
4     0.363182  0.334083  0.157363  0.069083  1.324083  0.100872
...        ...       ...       ...       ...       ...       ...
4860  0.218079  0.201750  0.118112  0.025333  0.869917  0.081284
4861  0.083927  0.073333  0.053302  0.019500  0.472750  0.054994
4862  0.130136  0.133083  0.037960  0.062250  0.218500  0.076693
4863  0.363157  0.194667  0.327350  0.070000  1.947583  0.085574
4864  0.108530  0.079625  0.071175  0.039583  0.698333  0.044884

[4865 rows x 6 columns]


In [31]:
kmedoids = KMedoids(n_clusters=20, random_state=42)
kmedoids.fit(X)

# Get the cluster assignments
kmedoid_clusters = kmedoids.predict(X)

# Add the cluster assignments to the dataframe
energy_stats['kmedoid_clusters'] = kmedoid_clusters
print(energy_stats['kmedoid_clusters'].value_counts())

dump(kmedoids, 'kmedoids_model.joblib')

print(energy_stats.head())

kmedoid_clusters
16    649
19    581
18    428
10    413
17    341
2     331
1     252
12    222
6     218
4     182
13    181
11    177
9     170
15    114
0     105
7     105
8     103
5     102
3     100
14     91
Name: count, dtype: int64
       LCLid      mean    median       std       min       max  gradient  \
0  MAC000002  0.252475  0.212667  0.158780  0.000000  1.654250  0.109262   
1  MAC000003  0.397265  0.239333  0.452749  0.023083  3.211750  0.176724   
2  MAC000005  0.095424  0.079083  0.062499  0.010417  0.878667  0.057801   
3  MAC000007  0.197888  0.170333  0.141215  0.038500  1.905833  0.105829   
4  MAC000008  0.363182  0.334083  0.157363  0.069083  1.324083  0.100872   

   kmedoid_clusters  
0                10  
1                 6  
2                 4  
3                10  
4                11  


In [32]:
# Get the cluster medoids and the corresponding households
cluster_medoids = kmedoids.cluster_centers_

# Find the id of the household that matches the medoid
medoid_households = []
for medoid in cluster_medoids:
    household_id = energy_stats.iloc[(energy_stats[['mean', 'median', 'std', 'min', 'max', 'gradient']].values == medoid)]['LCLid']
    household_id = household_id.values[0]
    medoid_households.append(household_id)

print(medoid_households)

['MAC000155', 'MAC003269', 'MAC003828', 'MAC004529', 'MAC000611', 'MAC004212', 'MAC004949', 'MAC001099', 'MAC000362', 'MAC000901', 'MAC000132', 'MAC003300', 'MAC001508', 'MAC000492', 'MAC005505', 'MAC001056', 'MAC000400', 'MAC001713', 'MAC000215', 'MAC000173']


In [62]:
fig = make_subplots(rows=5, cols=1, subplot_titles=[f'{household}' for household in medoid_households])

counter = 1
for household in medoid_households:
    if counter > 5:
        break
    row = counter
    col = 1
    df_household = smoothed_energy_df[smoothed_energy_df['LCLid'] == household][:480]
    color = colors[counter - 1]
    fig.add_trace(go.Scatter(x=df_household['tstp'], y=df_household['energy(kWh/hh)_smoothed'], mode='lines', name=household, line=dict(color=color), line_width=1), row=row, col=col)
    counter += 1

fig.update_layout(height=1000, width=900, title_text="Smoothed energy consumption profiles of 5 cluster medoids")
fig.update_layout(showlegend=False)
fig.update_yaxes(range=[0, 1])
fig.show()




# cluster = 1
# num_households = 5

# households_same_cluster = energy_stats[energy_stats['kmedoid_clusters'] == cluster]['LCLid']
# households_same_cluster = np.random.choice(households_same_cluster, num_households).tolist()

# fig = make_subplots(rows=5, cols=1, subplot_titles=[f'{household}' for household in households_same_cluster])

# for household in households_same_cluster:
#     index = households_same_cluster.index(household)
#     row = index + 1
#     col = 1
#     df_household = smoothed_energy_df[smoothed_energy_df['LCLid'] == household][:480]
#     fig.add_trace(go.Scatter(x=df_household['tstp'], y=df_household['energy(kWh/hh)_smoothed'], mode='lines', name=household, line_width = 1), row=row, col=col)

# fig.update_layout(height=1000, width=900, title_text=f"Smoothed energy consumption profiles of 5 households in Cluster {cluster}")
# fig.update_layout(showlegend=False)
# fig.update_yaxes(range=[0, 1])
# fig.show()

In [33]:
# fig = go.Figure()

# for household in example_households:
#     df_household = smoothed_energy_df[smoothed_energy_df['LCLid'] == household]
#     cluster = energy_stats[energy_stats['LCLid'] == household]['kmedoid_clusters'].values[0]
#     color = colors[cluster]
#     fig.add_trace(go.Scatter(x=df_household['tstp'], y=df_household['energy(kWh/hh)_smoothed'], mode='lines', name=household, line=dict(color=color), line_width=0.5))

# fig.layout.title = 'Energy consumption profiles of the example households'
# fig.show()

In [34]:
# households_100 = np.random.choice(households, 10)
# fig = go.Figure()

# for household in households_100:
#     df_household = smoothed_energy_df[smoothed_energy_df['LCLid'] == household]
#     cluster = energy_stats[energy_stats['LCLid'] == household]['kmedoid_clusters'].values[0]
#     color = colors[cluster]
#     fig.add_trace(go.Scatter(x=df_household['tstp'], y=df_household['energy(kWh/hh)_smoothed'], mode='lines', name=household, line=dict(color=color), line_width=0.5))

# fig.layout.title = 'Energy consumption profiles of 10 random households'
# fig.show()

In [35]:
# # Create a figure with a subplot grid of 5 rows by 2 columns
# fig = make_subplots(rows=5, cols=2, subplot_titles=[f'Cluster {i+1}' for i in range(10)])

# for cluster in range(10):
#     row = (cluster // 2) + 1
#     col = (cluster % 2) + 1
#     cluster_households = energy_stats[energy_stats['kmedoid_clusters'] == cluster]['LCLid']
#     households = np.random.choice(cluster_households, 3)
#     for household in households:
#         df_household = smoothed_energy_df[smoothed_energy_df['LCLid'] == household]
#         fig.add_trace(go.Scatter(x=df_household['tstp'], y=df_household['energy(kWh/hh)_smoothed'], mode='lines', name=household, line_width=1), row=row, col=col)

# fig.update_layout(height=1000, width=900, title_text="Energy consumption profiles of 10 households in each cluster")
# fig.update_layout(showlegend=False)
# fig.show()

In [36]:
# save the medoid households
energy_stats['is_medoid'] = energy_stats['LCLid'].isin(medoid_households).astype(int)
print(energy_stats.head())

       LCLid      mean    median       std       min       max  gradient  \
0  MAC000002  0.252475  0.212667  0.158780  0.000000  1.654250  0.109262   
1  MAC000003  0.397265  0.239333  0.452749  0.023083  3.211750  0.176724   
2  MAC000005  0.095424  0.079083  0.062499  0.010417  0.878667  0.057801   
3  MAC000007  0.197888  0.170333  0.141215  0.038500  1.905833  0.105829   
4  MAC000008  0.363182  0.334083  0.157363  0.069083  1.324083  0.100872   

   kmedoid_clusters  is_medoid  
0                10          0  
1                 6          0  
2                 4          0  
3                10          0  
4                11          0  


In [ ]:
# cluster = 0
# num_households = 20

# households_same_cluster = energy_stats[energy_stats['kmedoid_clusters'] == cluster]['LCLid']
# households_same_cluster = noice(households_same_cluster, num_households).tolist()p.random.ch

# fig = make_subplots(rows=5, cols=4, subplot_titles=[f'{household}' for household in households_same_cluster])

# for household in households_same_cluster:
#     index = households_same_cluster.index(household)
#     row = (index // 2) + 1
#     col = (index % 2) + 1
#     df_household = smoothed_energy_df[smoothed_energy_df['LCLid'] == household]
#     fig.add_trace(go.Scatter(x=df_household['tstp'], y=df_household['energy(kWh/hh)_smoothed'], mode='lines', name=household, line_width = 1), row=row, col=col)

# fig.update_layout(title_text=f"Smoothed energy consumption profiles of 10 medoid households Cluster {cluster}")
# fig.update_layout(showlegend=False)
# fig.update_yaxes(range=[0, 1])
# fig.show()

In [58]:
cluster = 1
num_households = 5

households_same_cluster = energy_stats[energy_stats['kmedoid_clusters'] == cluster]['LCLid']
households_same_cluster = np.random.choice(households_same_cluster, num_households).tolist()

fig = make_subplots(rows=5, cols=1, subplot_titles=[f'{household}' for household in households_same_cluster])

for household in households_same_cluster:
    index = households_same_cluster.index(household)
    row = index + 1
    col = 1
    df_household = smoothed_energy_df[smoothed_energy_df['LCLid'] == household][:480]
    fig.add_trace(go.Scatter(x=df_household['tstp'], y=df_household['energy(kWh/hh)_smoothed'], mode='lines', name=household, line_width = 1), row=row, col=col)

fig.update_layout(height=1000, width=900, title_text=f"Smoothed energy consumption profiles of 5 households in Cluster {cluster}")
fig.update_layout(showlegend=False)
fig.update_yaxes(range=[0, 1])
fig.show()

In [38]:
# cluster = 1
# num_households = 10

# households_same_cluster = energy_stats[energy_stats['kmedoid_clusters'] == cluster]['LCLid']
# households_same_cluster = np.random.choice(households_same_cluster, num_households).tolist()

# fig = make_subplots(rows=5, cols=2, subplot_titles=[f'{household}' for household in households_same_cluster])

# for household in households_same_cluster:
#     index = households_same_cluster.index(household)
#     row = (index // 2) + 1
#     col = (index % 2) + 1
#     df_household = smoothed_energy_df[smoothed_energy_df['LCLid'] == household]
#     fig.add_trace(go.Scatter(x=df_household['tstp'], y=df_household['energy(kWh/hh)_smoothed'], mode='lines', name=household, line_width = 1), row=row, col=col)

# fig.update_layout(height=1000, width=900, title_text=f"Smoothed energy consumption profiles of 10 medoid households Cluster {cluster}")
# fig.update_layout(showlegend=False)
# fig.update_yaxes(range=[0, 1])
# fig.show()

In [39]:
# cluster = 2
# num_households = 10

# households_same_cluster = energy_stats[energy_stats['kmedoid_clusters'] == cluster]['LCLid']
# households_same_cluster = np.random.choice(households_same_cluster, num_households).tolist()

# fig = make_subplots(rows=5, cols=2, subplot_titles=[f'{household}' for household in households_same_cluster])

# for household in households_same_cluster:
#     index = households_same_cluster.index(household)
#     row = (index // 2) + 1
#     col = (index % 2) + 1
#     df_household = smoothed_energy_df[smoothed_energy_df['LCLid'] == household]
#     fig.add_trace(go.Scatter(x=df_household['tstp'], y=df_household['energy(kWh/hh)_smoothed'], mode='lines', name=household, line_width = 1), row=row, col=col)

# fig.update_layout(height=1000, width=900, title_text=f"Smoothed energy consumption profiles of 10 medoid households Cluster {cluster}")
# fig.update_layout(showlegend=False)
# fig.update_yaxes(range=[0, 1])
# fig.show()

# Modify the statistical

In [27]:
energy_stats2 = energy_df.groupby('LCLid')['energy(kWh/hh)'].agg(['mean', 'median', 'std', 'min', 'max'])

# Turn it into a dataframe 
energy_stats2 = energy_stats2.reset_index()
energy_stats2 = energy_stats2.merge(mean_gradient_df, on='LCLid', how='left')

energy_stats2.head()

,LCLid,mean,median,std,min,max,gradient
0,MAC000002,0.252511,0.158,0.247080,0.000,2.994,0.109262
1,MAC000003,0.397263,0.166,0.614692,0.007,3.921,0.176724
2,MAC000005,0.095425,0.041,0.122226,0.010,1.979,0.057801
3,MAC000007,0.197888,0.115,0.234394,0.015,3.784,0.105829
4,MAC000008,0.363184,0.296,0.241725,0.000,3.581,0.100872


In [29]:
print(energy_stats.head())

       LCLid      mean    median       std       min       max  gradient  \
0  MAC000002  0.252475  0.212667  0.158780  0.000000  1.654250  0.109262   
1  MAC000003  0.397265  0.239333  0.452749  0.023083  3.211750  0.176724   
2  MAC000005  0.095424  0.079083  0.062499  0.010417  0.878667  0.057801   
3  MAC000007  0.197888  0.170333  0.141215  0.038500  1.905833  0.105829   
4  MAC000008  0.363182  0.334083  0.157363  0.069083  1.324083  0.100872   

   kmedoid_clusters  is_medoid  
0                10          0  
1                 6          0  
2                 4          0  
3                10          0  
4                11          0  


In [31]:
# Add the last two columns to the dataframe
energy_stats2['kmedoid_clusters'] = energy_stats['kmedoid_clusters']
energy_stats2['is_medoid'] = energy_stats['is_medoid']

print(energy_stats2.head())

energy_stats = energy_stats2

       LCLid      mean  median       std    min    max  gradient  \
0  MAC000002  0.252511   0.158  0.247080  0.000  2.994  0.109262   
1  MAC000003  0.397263   0.166  0.614692  0.007  3.921  0.176724   
2  MAC000005  0.095425   0.041  0.122226  0.010  1.979  0.057801   
3  MAC000007  0.197888   0.115  0.234394  0.015  3.784  0.105829   
4  MAC000008  0.363184   0.296  0.241725  0.000  3.581  0.100872   

   kmedoid_clusters  is_medoid  
0                10          0  
1                 6          0  
2                 4          0  
3                10          0  
4                11          0  


# Spikes data

In [32]:
spike_df = pd.read_csv('../../Data_processed/energy_df_spike_type.csv')
print(spike_df.head())

       LCLid                 tstp  spike_type  spike_magnitude
0  MAC000002  2012-10-12 00:30:00           0              1.0
1  MAC000002  2012-10-12 01:00:00           0              1.0
2  MAC000002  2012-10-12 01:30:00           0              1.0
3  MAC000002  2012-10-12 02:00:00           0              1.0
4  MAC000002  2012-10-12 02:30:00           0              1.0


In [33]:
spike_df['spike_magnitude'].dtype

dtype('float64')

In [34]:
spike_df['spike_magnitude'] = spike_df['spike_magnitude']
spike_df['spike_magnitude'].describe()

count    1.472021e+08
mean     9.639485e-01
std      1.840984e-01
min      2.300000e-02
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.076100e+01
Name: spike_magnitude, dtype: float64

# Weather Data

In [35]:
weather_point_df = utils.get_weather_df()

weather_point_df = utils.process_weather_df(weather_point_df)

print(weather_point_df.shape)
weather_point_df[0:25]

(42333, 4)


,tstp,temperature,windSpeed,humidity
0,2011-11-01 00:00:00,0.50290,0.20800,0.84420
1,2011-11-01 00:30:00,0.49290,0.20700,0.87665
2,2011-11-01 01:00:00,0.48290,0.20600,0.90910
3,2011-11-01 01:30:00,0.49500,0.22730,0.89610
4,2011-11-01 02:00:00,0.50710,0.24860,0.88310
5,2011-11-01 02:30:00,0.51340,0.25675,0.86365
6,2011-11-01 03:00:00,0.51970,0.26490,0.84420
7,2011-11-01 03:30:00,0.52025,0.26560,0.85715
8,2011-11-01 04:00:00,0.52080,0.26630,0.87010
9,2011-11-01 04:30:00,0.52130,0.25915,0.87010


# Merge the data

In [36]:
print(weather_point_df.isna().sum())

tstp           0
temperature    0
windSpeed      0
humidity       0
dtype: int64


In [37]:
complete_df = smoothed_energy_df.merge(energy_df, on=['LCLid', 'tstp'], how='left')
complete_df = complete_df.merge(energy_stats, on='LCLid', how='left')

complete_df.head()

,LCLid,tstp,energy(kWh/hh)_smoothed,energy(kWh/hh),mean,median,std,min,max,gradient,kmedoid_clusters,is_medoid
0,MAC000002,2012-10-12 00:30:00,0.0,0.0,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0
1,MAC000002,2012-10-12 01:00:00,0.0,0.0,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0
2,MAC000002,2012-10-12 01:30:00,0.0,0.0,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0
3,MAC000002,2012-10-12 02:00:00,0.0,0.0,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0
4,MAC000002,2012-10-12 02:30:00,0.0,0.0,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0


In [38]:
complete_df['tstp'] = pd.to_datetime(complete_df['tstp'])
spike_df['tstp'] = pd.to_datetime(spike_df['tstp'])
complete_df = complete_df.merge(spike_df, on=['LCLid', 'tstp'], how='left')
complete_df.head()

,LCLid,tstp,energy(kWh/hh)_smoothed,energy(kWh/hh),mean,median,std,min,max,gradient,kmedoid_clusters,is_medoid,spike_type,spike_magnitude
0,MAC000002,2012-10-12 00:30:00,0.0,0.0,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0,0,1.0
1,MAC000002,2012-10-12 01:00:00,0.0,0.0,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0,0,1.0
2,MAC000002,2012-10-12 01:30:00,0.0,0.0,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0,0,1.0
3,MAC000002,2012-10-12 02:00:00,0.0,0.0,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0,0,1.0
4,MAC000002,2012-10-12 02:30:00,0.0,0.0,0.252511,0.158,0.24708,0.0,2.994,0.109262,10,0,0,1.0


In [39]:
complete_df['spike_magnitude'].dtype

dtype('float64')

In [40]:
print(complete_df.isna().sum())

LCLid                      0
tstp                       0
energy(kWh/hh)_smoothed    0
energy(kWh/hh)             0
mean                       0
median                     0
std                        0
min                        0
max                        0
gradient                   0
kmedoid_clusters           0
is_medoid                  0
spike_type                 0
spike_magnitude            0
dtype: int64


In [41]:
complete_df['tstp'] = pd.to_datetime(complete_df['tstp'])
weather_point_df['tstp'] = pd.to_datetime(weather_point_df['tstp'])
merged_df = pd.merge(complete_df, weather_point_df, on=['tstp'], how = 'left')

print(merged_df.head())
print(merged_df.isna().sum())

       LCLid                tstp  energy(kWh/hh)_smoothed  energy(kWh/hh)  \
0  MAC000002 2012-10-12 00:30:00                      0.0             0.0   
1  MAC000002 2012-10-12 01:00:00                      0.0             0.0   
2  MAC000002 2012-10-12 01:30:00                      0.0             0.0   
3  MAC000002 2012-10-12 02:00:00                      0.0             0.0   
4  MAC000002 2012-10-12 02:30:00                      0.0             0.0   

       mean  median      std  min    max  gradient  kmedoid_clusters  \
0  0.252511   0.158  0.24708  0.0  2.994  0.109262                10   
1  0.252511   0.158  0.24708  0.0  2.994  0.109262                10   
2  0.252511   0.158  0.24708  0.0  2.994  0.109262                10   
3  0.252511   0.158  0.24708  0.0  2.994  0.109262                10   
4  0.252511   0.158  0.24708  0.0  2.994  0.109262                10   

   is_medoid  spike_type  spike_magnitude  temperature  windSpeed  humidity  
0          0           0  

In [42]:
merged_df.fillna(method='ffill', inplace=True)

In [43]:
print(merged_df.isna().sum())

LCLid                      0
tstp                       0
energy(kWh/hh)_smoothed    0
energy(kWh/hh)             0
mean                       0
median                     0
std                        0
min                        0
max                        0
gradient                   0
kmedoid_clusters           0
is_medoid                  0
spike_type                 0
spike_magnitude            0
temperature                0
windSpeed                  0
humidity                   0
dtype: int64


In [44]:
merged_df['tstp'] = pd.to_datetime(merged_df['tstp'])
    
# Filter data to only include rows where the month and day fall within the summer range of 2013
df_filtered = merged_df[(merged_df['tstp'] >= '2013-01-01') & (merged_df['tstp'] < '2014-01-01')]


In [45]:
df_filtered.to_csv('../../Data_processed/Kmedoids_with_Weather_Espike.csv', index=False)